In [1]:
import pandas as pd
import numpy as np

tx_cols = ["TransactionID", "isFraud", "TransactionDT", "TransactionAmt",
           "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
           "addr1", "dist1", "P_emaildomain", "R_emaildomain"]
id_cols = ["TransactionID", "DeviceType", "DeviceInfo"]

tx = pd.read_csv("data/train_transaction.csv", usecols=tx_cols)
idt = pd.read_csv("data/train_identity.csv", usecols=id_cols)
df = tx.merge(idt, on="TransactionID", how="left")

# sort by time so "before" really means before
df = df.sort_values("TransactionDT").reset_index(drop=True)
print(df.shape)

(590540, 17)


In [2]:
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["day_of_week"] = (df["TransactionDT"] // 86400) % 7

df["card_key"] = (df["card1"].astype(str) + "_" + df["card2"].astype(str) + "_" +
                  df["card3"].astype(str) + "_" + df["card5"].astype(str))

print(df["card_key"].nunique(), "unique cards")

14845 unique cards


In [3]:
g = df.groupby("card_key")

# velocity
df["tx_count_before"] = g.cumcount()
df["secs_since_last"] = g["TransactionDT"].diff()

# amount compared with this card's usual amount (past only)
past_sum = g["TransactionAmt"].cumsum() - df["TransactionAmt"]
df["avg_amt_before"] = past_sum / df["tx_count_before"].replace(0, np.nan)
df["amt_vs_avg"] = df["TransactionAmt"] / df["avg_amt_before"]

# did something change since the last transaction of this card?
for col, new in [("DeviceInfo", "device_changed"),
                 ("addr1", "addr_changed"),
                 ("P_emaildomain", "email_changed")]:
    prev = g[col].shift(1)
    df[new] = ((df[col] != prev) & df[col].notna() & prev.notna()).astype(int)

print(df[["tx_count_before", "secs_since_last", "amt_vs_avg",
          "device_changed", "addr_changed", "email_changed"]].describe())

       tx_count_before  secs_since_last     amt_vs_avg  device_changed  \
count    590540.000000     5.756950e+05  575695.000000   590540.000000   
mean       1136.236927     1.728830e+05       1.088528        0.063754   
std        1978.491703     7.159087e+05       1.733773        0.244314   
min           0.000000     0.000000e+00       0.002841        0.000000   
25%          39.000000     9.900000e+02       0.391412        0.000000   
50%         284.000000     6.001000e+03       0.659707        0.000000   
75%        1282.000000     5.905950e+04       1.159798        0.000000   
max       14111.000000     1.559967e+07     145.944802        1.000000   

        addr_changed  email_changed  
count  590540.000000  590540.000000  
mean        0.457256       0.407505  
std         0.498170       0.491371  
min         0.000000       0.000000  
25%         0.000000       0.000000  
50%         0.000000       0.000000  
75%         1.000000       1.000000  
max         1.000000       1.

In [4]:
for c in ["device_changed", "addr_changed", "email_changed"]:
    print(df.groupby(c)["isFraud"].agg(["mean", "count"]))
    print()

df["fast"] = (df["secs_since_last"] < 60).astype(int)
print(df.groupby("fast")["isFraud"].agg(["mean", "count"]))

                    mean   count
device_changed                  
0               0.032987  552891
1               0.064411   37649

                  mean   count
addr_changed                  
0             0.045006  320512
1             0.023101  270028

                   mean   count
email_changed                  
0              0.037672  349892
1              0.031091  240648

          mean   count
fast                  
0     0.034009  571295
1     0.064121   19245


In [5]:
df["event_timestamp"] = pd.Timestamp("2017-12-01") + pd.to_timedelta(df["TransactionDT"], unit="s")

feature_cols = ["TransactionID", "card_key", "event_timestamp", "hour", "day_of_week",
                "tx_count_before", "secs_since_last", "amt_vs_avg",
                "device_changed", "addr_changed", "email_changed"]

df[feature_cols + ["isFraud"]].to_parquet("data/features.parquet", index=False)
print("saved")

saved


In [6]:
import pandas as pd
f = pd.read_parquet("data/features.parquet")
print(f.shape)
print(f.head())

(590540, 12)
   TransactionID                 card_key     event_timestamp  hour  \
0        2987000    13926_nan_150.0_142.0 2017-12-02 00:00:00     0   
1        2987001   2755_404.0_150.0_102.0 2017-12-02 00:00:01     0   
2        2987002   4663_490.0_150.0_166.0 2017-12-02 00:01:09     0   
3        2987003  18132_567.0_150.0_117.0 2017-12-02 00:01:39     0   
4        2987004   4497_514.0_150.0_102.0 2017-12-02 00:01:46     0   

   day_of_week  tx_count_before  secs_since_last  amt_vs_avg  device_changed  \
0            1                0              NaN         NaN               0   
1            1                0              NaN         NaN               0   
2            1                0              NaN         NaN               0   
3            1                0              NaN         NaN               0   
4            1                0              NaN         NaN               0   

   addr_changed  email_changed  isFraud  
0             0              0       

In [7]:
import pandas as pd

f = pd.read_parquet("data/features.parquet")
f["card_id"], uniques = pd.factorize(f["card_key"])
f["card_id"] = f["card_id"].astype("int64")

f.to_parquet("data/features.parquet", index=False)
pd.DataFrame({"card_id": range(len(uniques)), "card_key": uniques}).to_parquet("data/card_map.parquet", index=False)

print(f[["card_key", "card_id"]].head())
print(f["card_id"].nunique(), "unique cards")

                  card_key  card_id
0    13926_nan_150.0_142.0        0
1   2755_404.0_150.0_102.0        1
2   4663_490.0_150.0_166.0        2
3  18132_567.0_150.0_117.0        3
4   4497_514.0_150.0_102.0        4
14845 unique cards


In [8]:
import os
os.makedirs("feature_repo", exist_ok=True)

In [9]:
%%writefile feature_repo/feature_store.yaml
project: fraud_detection
registry: registry.db
provider: local
online_store:
  type: sqlite
  path: online_store.db
entity_key_serialization_version: 3

Overwriting feature_repo/feature_store.yaml


In [10]:
%%writefile feature_repo/features.py
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource, ValueType
from feast.types import Float64, Int64

card = Entity(name="card_id", join_keys=["card_id"], value_type=ValueType.INT64)

source = FileSource(
    path="../data/features.parquet",
    timestamp_field="event_timestamp",
)

card_features = FeatureView(
    name="card_features",
    entities=[card],
    ttl=timedelta(days=365),
    schema=[
        Field(name="hour", dtype=Int64),
        Field(name="day_of_week", dtype=Int64),
        Field(name="tx_count_before", dtype=Int64),
        Field(name="secs_since_last", dtype=Float64),
        Field(name="amt_vs_avg", dtype=Float64),
        Field(name="device_changed", dtype=Int64),
        Field(name="addr_changed", dtype=Int64),
        Field(name="email_changed", dtype=Int64),
    ],
    source=source,
)

Overwriting feature_repo/features.py


In [11]:
import pandas as pd
from feast import FeatureStore

f = pd.read_parquet("data/features.parquet")
store = FeatureStore(repo_path="feature_repo")
some_card = int(f["card_id"].iloc[1000])

result = store.get_online_features(
    features=["card_features:tx_count_before", "card_features:amt_vs_avg",
              "card_features:device_changed", "card_features:secs_since_last"],
    entity_rows=[{"card_id": some_card}],
).to_dict()
print(result)

{'card_id': [299], 'secs_since_last': [725.0], 'device_changed': [0], 'tx_count_before': [6046], 'amt_vs_avg': [0.3835021129828901]}
